# Insertion, selections, and rigid transforms

`Recipe.insert()` records a manufacturing activation operation. The
collection is transformed once, packed into the eventual device world,
and remains dormant until its operation is reached.

In [ ]:
import tangle
from tangle.units import mm, um

# This collection is authored around a local origin and is not yet in
# the periodic simulation cell.
cell = tangle.Cell([1 * mm, 1 * mm, 2 * mm], periodic="xy")
material = tangle.Material("fiber", diameter=19 * um)
collection = tangle.FiberCollection.from_centerlines(
    [[[-0.3 * mm, 0.0, 0.0], [0.3 * mm, 0.0, 0.0]]],
    material,
    name="local-coordinate ply",
    formation_layer=0,
)
# The recipe takes its stack axis (z) from the cell; pass
# stack_axis="x" (or 0) to Recipe to override it.
recipe = tangle.Recipe(cell)
# Rotation is applied first, then translation places the rotated fiber.
selection = recipe.insert(
    collection,
    name="placed ply",
    translation=[0.5 * mm, 0.5 * mm, 0.4 * mm],
    rotation=[[0.0, -1.0, 0.0],
              [1.0,  0.0, 0.0],
              [0.0,  0.0, 1.0]],
)
print(recipe.stack_axis, selection.name, selection.fiber_ids, selection.formation_step)

`name` labels the returned `FiberSelection`; `translation` is in meters;
and `rotation` is a 3×3 matrix applied before translation. Selections are
stable handles for reporting and future selection-scoped APIs. Use
`recipe.operations()` to audit ordering before a costly run and
`recipe.centerlines()` to inspect the packed initial geometry.

A `Recipe` can also start from an existing `Assembly`, such as the
`result.assembly` of an earlier run, to continue manufacturing from
relaxed geometry.

In [ ]:
# Methods append ordered operations; no solver launches until run().
recipe.relax_until_converged(max_iterations=2_000)
print(*recipe.operations(), sep="\n")
recipe.centerlines()